# NB16 — Strict Outer-Training Calibration and Selective Prediction

Final reliability analysis used in the manuscript. For every outer-test fold, temperature scaling and selective-prediction thresholds are estimated exclusively from internal OOF predictions generated inside the corresponding outer-training partition. The outer-test fold does not contribute labels or predictions to its own calibration or rejection threshold.

Outputs include NLL and multiclass Brier score with grain-cluster bootstrap confidence intervals, equal-width and adaptive equal-mass ECE, class-conditional ECE, high-confidence bins (0.99–1.00), AURC, exact Clopper–Pearson intervals, rejected-set composition, and TabPFN/LDA-XGBoost ROC curves.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install "tabpfn==9.0.0" xgboost scipy scikit-learn pandas openpyxl matplotlib
from pathlib import Path
from getpass import getpass
import os, json, math, gc, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.stats import beta
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import f1_score, accuracy_score, log_loss, roc_curve, auc
from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier
import tabpfn
warnings.filterwarnings('ignore')

ROOT=Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATASET=ROOT/'01_DATA'/'INIAP_Dataset.xlsx'
OUT=ROOT/'03_RESULTS'/'NB16_STRICT_CALIBRATION'
TAB=ROOT/'06_TABLES'/'NB16_STRICT_CALIBRATION'
FIG=ROOT/'05_FIGURES'/'NB16_STRICT_CALIBRATION'
CKPT=OUT/'checkpoints'
for p in [OUT,TAB,FIG,CKPT]: p.mkdir(parents=True,exist_ok=True)

token=os.environ.get('TABPFN_TOKEN','').strip()
if not token:
    try: token=str(userdata.get('TABPFN_TOKEN') or '').strip()
    except Exception: token=''
if not token: token=getpass('Prior Labs API key (hidden): ').strip()
if not token: raise RuntimeError('TABPFN_TOKEN is required for TabPFN access.')
os.environ['TABPFN_TOKEN']=token

df=pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
FEATURES=['Area','Perimeter','MajorAxisLength','MinorAxisLength','AspectRatio','ConvexArea','EquivDiameter','Extent','Solidity','roundness','Compactness']
X=df[FEATURES].copy(); le=LabelEncoder(); y=pd.Series(le.fit_transform(df['Class'].astype(str)),index=df.index)
CLASSES=list(le.classes_); K=len(CLASSES)
SEEDS=[2026,2027,2028]; OUTER_FOLDS=5; INNER_CAL_FOLDS=3; TARGET_COVERAGES=[1,.95,.90,.80,.70,.60,.50]
BOOTSTRAPS=3000; RNG=np.random.default_rng(20260929)
NB09_OOF=ROOT/'03_RESULTS'/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'all_oof_harmonized.csv'
assert DATASET.exists() and NB09_OOF.exists()
existing_oof=pd.read_csv(NB09_OOF)
print('TabPFN',getattr(tabpfn,'__version__','unknown'),'| classes:',CLASSES)


In [ ]:
def norm_probs(P,eps=1e-12):
    P=np.asarray(P,float); P=np.clip(P,eps,None); return P/P.sum(1,keepdims=True)
def temp_transform(P,T):
    Z=np.log(norm_probs(P))/float(T); Z-=Z.max(1,keepdims=True); E=np.exp(Z); return E/E.sum(1,keepdims=True)
def fit_temperature(ytrue,P):
    def obj(logT): return log_loss(ytrue,temp_transform(P,np.exp(logT)),labels=np.arange(K))
    return float(np.exp(minimize_scalar(obj,bounds=(math.log(.05),math.log(20)),method='bounded').x))
def brier(ytrue,P):
    Y=np.eye(K)[np.asarray(ytrue,int)]; return float(np.mean(np.sum((norm_probs(P)-Y)**2,axis=1)))
def top_info(ytrue,P):
    P=norm_probs(P); pred=P.argmax(1); conf=P.max(1); return pred,conf,(pred==np.asarray(ytrue,int)).astype(int)
def adaptive_ece(ytrue,P,n_bins=10):
    _,conf,corr=top_info(ytrue,P); chunks=np.array_split(np.argsort(conf),n_bins); e=0.; rows=[]
    for b,idx in enumerate(chunks,1):
        mc=float(conf[idx].mean()); ac=float(corr[idx].mean()); n=len(idx); e+=(n/len(conf))*abs(ac-mc); rows.append([b,n,mc,ac,abs(ac-mc)])
    return float(e),pd.DataFrame(rows,columns=['bin','n','mean_confidence','accuracy','gap'])
def equal_width_ece(ytrue,P,n_bins=10):
    _,conf,corr=top_info(ytrue,P); edges=np.linspace(0,1,n_bins+1); e=0.; rows=[]
    for b in range(n_bins):
        m=(conf>=edges[b])&((conf<edges[b+1]) if b<n_bins-1 else (conf<=edges[b+1]))
        if not m.any(): continue
        mc=float(conf[m].mean()); ac=float(corr[m].mean()); n=int(m.sum()); e+=(n/len(conf))*abs(ac-mc); rows.append([b+1,edges[b],edges[b+1],n,mc,ac,abs(ac-mc)])
    return float(e),pd.DataFrame(rows,columns=['bin','lower','upper','n','mean_confidence','accuracy','gap'])
def cp_ci(correct,n,alpha=.05):
    lo=0 if correct==0 else beta.ppf(alpha/2,correct,n-correct+1); hi=1 if correct==n else beta.ppf(1-alpha/2,correct+1,n-correct); return float(lo),float(hi)
def aurc(ytrue,P):
    _,conf,corr=top_info(ytrue,P); order=np.argsort(-conf); risk=np.cumsum(1-corr[order])/np.arange(1,len(corr)+1); cov=np.arange(1,len(corr)+1)/len(corr); return float(np.trapz(np.r_[0,risk],np.r_[0,cov]))


In [ ]:
def lda_xgb(seed):
    return Pipeline([('scale',StandardScaler()),('dr',LinearDiscriminantAnalysis()),('clf',XGBClassifier(objective='multi:softprob',eval_metric='mlogloss',tree_method='hist',n_jobs=1,random_state=seed))])
GRID={'dr__n_components':[1,2,3],'clf__n_estimators':[300,600],'clf__max_depth':[3,5],'clf__learning_rate':[.03,.1]}
def internal_oof_lda_xgb(Xtr,ytr,seed,fold):
    cv=StratifiedKFold(3,shuffle=True,random_state=seed+fold)
    gs=GridSearchCV(lda_xgb(seed+fold),GRID,scoring='f1_macro',cv=cv,n_jobs=-1,refit=True); gs.fit(Xtr,ytr)
    inner=StratifiedKFold(INNER_CAL_FOLDS,shuffle=True,random_state=seed+100+fold)
    P=cross_val_predict(lda_xgb(seed+fold).set_params(**gs.best_params_),Xtr,ytr,cv=inner,method='predict_proba',n_jobs=1)
    return P,gs.best_params_
def internal_oof_tabpfn(Xtr,ytr,seed,fold):
    inner=StratifiedKFold(INNER_CAL_FOLDS,shuffle=True,random_state=seed+100+fold); P=np.zeros((len(Xtr),K))
    for tr,va in inner.split(Xtr,ytr):
        m=TabPFNClassifier(device='cuda'); m.fit(Xtr.iloc[tr].to_numpy(np.float32),ytr.iloc[tr].to_numpy()); P[va]=m.predict_proba(Xtr.iloc[va].to_numpy(np.float32)); del m; gc.collect()
    return P
oof=existing_oof[existing_oof.feature_set.astype(str).eq('FULL_11')].copy(); oof['model_norm']=oof.model.astype(str).str.replace('-','_',regex=False)
def saved_outer(model,seed,fold):
    g=oof[(oof.model_norm==model)&(oof.seed==seed)&(oof.outer_fold==fold)].copy(); assert len(g)==800
    cols=[f'prob_{c}' for c in CLASSES]; return g.row_id.to_numpy(int),le.transform(g.y_true.astype(str)),g[cols].to_numpy(float)


In [ ]:
raw_parts=[]; cal_parts=[]; temp_rows=[]; threshold_rows=[]
for seed in SEEDS:
    outer=StratifiedKFold(OUTER_FOLDS,shuffle=True,random_state=seed)
    for fold,(tr,te) in enumerate(outer.split(X,y),1):
        Xtr=X.iloc[tr].reset_index(drop=True); ytr=y.iloc[tr].reset_index(drop=True)
        for model in ['LDA_XGBoost','TabPFN']:
            rid,ytest,Ptest=saved_outer(model,seed,fold)
            Pcal,bp=(internal_oof_lda_xgb(Xtr,ytr,seed,fold) if model=='LDA_XGBoost' else (internal_oof_tabpfn(Xtr,ytr,seed,fold),{}))
            T=fit_temperature(ytr.to_numpy(),Pcal); PcalT=temp_transform(Pcal,T); PtestT=temp_transform(Ptest,T)
            _,conf_cal,_=top_info(ytr.to_numpy(),PcalT)
            temp_rows.append([model,seed,fold,T])
            for cov in TARGET_COVERAGES:
                tau=-np.inf if cov==1 else float(np.quantile(conf_cal,1-cov,method='higher')); threshold_rows.append([model,seed,fold,cov,tau])
            for i,rowid in enumerate(rid):
                a={'row_id':int(rowid),'seed':seed,'outer_fold':fold,'model':model,'y_true':int(ytest[i])}; b=a.copy()
                for k,c in enumerate(CLASSES): a[f'prob_{c}']=Ptest[i,k]; b[f'prob_{c}']=PtestT[i,k]
                raw_parts.append(a); cal_parts.append(b)
raw=pd.DataFrame(raw_parts); strict=pd.DataFrame(cal_parts); temps=pd.DataFrame(temp_rows,columns=['model','seed','outer_fold','temperature']); thresholds=pd.DataFrame(threshold_rows,columns=['model','seed','outer_fold','target_coverage','threshold'])
raw.to_csv(OUT/'strict_outer_raw_predictions.csv',index=False); strict.to_csv(OUT/'strict_outer_calibrated_predictions.csv',index=False); temps.to_csv(TAB/'strict_temperature_by_outer_fold.csv',index=False); thresholds.to_csv(TAB/'strict_thresholds_by_outer_fold.csv',index=False)


In [ ]:
prob_cols=[f'prob_{c}' for c in CLASSES]; metric_rows=[]; sel_rows=[]; rejected_rows=[]
for (model,seed),g in strict.groupby(['model','seed']):
    P=g[prob_cols].to_numpy(); yt=g.y_true.to_numpy(int); ae,_=adaptive_ece(yt,P); ee,_=equal_width_ece(yt,P)
    metric_rows.append([model,seed,log_loss(yt,P,labels=np.arange(K)),brier(yt,P),ee,ae,aurc(yt,P)])
metrics=pd.DataFrame(metric_rows,columns=['model','seed','nll','brier','ece_equal_width_10','ece_adaptive_equal_mass_10','aurc']); metrics.to_csv(TAB/'strict_calibration_metrics_by_seed.csv',index=False)
for (model,seed,fold),g in strict.groupby(['model','seed','outer_fold']):
    P=g[prob_cols].to_numpy(); pred,conf,corr=top_info(g.y_true.to_numpy(int),P)
    for cov in TARGET_COVERAGES:
        tau=thresholds.query('model==@model and seed==@seed and outer_fold==@fold and target_coverage==@cov').threshold.iloc[0]; keep=np.ones(len(g),bool) if not np.isfinite(tau) else conf>=tau
        n=int(keep.sum()); correct=int(corr[keep].sum()); lo,hi=cp_ci(correct,n)
        sel_rows.append([model,seed,fold,cov,float(tau),n,len(g)-n,n-correct,n/len(g),correct/n,lo,hi])
        for k,c in enumerate(CLASSES): rejected_rows.append([model,seed,fold,cov,c,int(((~keep)&(g.y_true.to_numpy(int)==k)).sum())])
sel=pd.DataFrame(sel_rows,columns=['model','seed','outer_fold','target_coverage','threshold','n_accepted','n_rejected','accepted_errors','realized_coverage','accepted_accuracy','accuracy_cp95_low','accuracy_cp95_high']); sel.to_csv(TAB/'selective_prediction_counts_ci_by_fold.csv',index=False)
pd.DataFrame(rejected_rows,columns=['model','seed','outer_fold','target_coverage','cultivar','rejected_n']).to_csv(TAB/'rejected_composition_by_cultivar_fold.csv',index=False)
display(metrics); display(sel[sel.target_coverage.isin([.9,.8])].head())
